# Code Review Assistant on Kaggle

Static analysis finds problems and says exactly where they are. A local code model then explains why each one matters, proposes a fix, and looks for logic errors no rule can see. This notebook runs the whole pipeline, measures it, compares two model sizes, and does the slow parts in parallel.

**Before you run it** (right-hand panel on Kaggle)

1. **Accelerator**: GPU T4 x2. On Colab, a single T4 is fine.
2. **Internet**: On.
3. **Add Input**: upload `code_review_assistant.py` as a dataset, or set `REPO_URL` below.
4. Press **Save Version, then Save & Run All (Commit)**. It runs on Kaggle's servers, so you can close the tab. Results land in the version's **Output** tab.

**How the time is used**

| Lane | Work | Uses | Runs |
| --- | --- | --- | --- |
| A | Download both models | network | first, in the background |
| B | Self-test, static evaluation, stdlib benchmark on every core, ruff and bandit checks | CPU | while A downloads |
| C | Smoke test, then score 3B and 7B with the corpus split across both GPUs | GPU | when A and B are done |
| D | Rating sheet, batch review of real files, UI check | GPU + CPU | last |

## 0. Settings

The only cell you should need to edit.

In [1]:
REPO_URL = "https://github.com/hansineesoundararajan/code-review-assistant.git"

LLM = "hf"                                   # "hf" runs a local model. "offline" dry-runs the plumbing with no model.
MODEL = "Qwen/Qwen2.5-Coder-3B-Instruct"     # about 6 GB in fp16, fits one T4 with room to spare

RUN_7B = True                                # also score the 7B model in 4-bit, for the model-selection comparison
MODEL_7B = "Qwen/Qwen2.5-Coder-7B-Instruct"
RUN_BATCH_DEMO = True                        # review a few real stdlib files with the model
RUN_UI_LIVE = False                          # True only in an interactive session: the live UI never returns

## 1. Environment

In [2]:
import csv
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

# Keep the log readable: no per-tensor loading bars, no repeated library notices.
# Real errors still print.
os.environ.update({
    "HF_HUB_DISABLE_PROGRESS_BARS": "1",
    "HF_HUB_VERBOSITY": "error",
    "TRANSFORMERS_VERBOSITY": "error",
    "TOKENIZERS_PARALLELISM": "false",
})

T0 = time.time()
TIMELINE = []


def sh(cmd, env=None):
    """Run a shell command and stream its output into the cell.

    Output from a plain os.system call can end up in the server log instead of
    the cell when a notebook runs headless, so the pipe is read here.
    """
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, env={**os.environ, **(env or {})})
    for line in proc.stdout:
        print(line, end="")
    return proc.wait()


def stamp(label):
    TIMELINE.append((time.time() - T0, label))
    print(f"[{time.time() - T0:5.0f}s] {label}")


def reachable(url):
    try:
        urllib.request.urlopen(url, timeout=5)
        return True
    except Exception:
        return False


ON_KAGGLE = Path("/kaggle").exists()
ON_COLAB = "google.colab" in sys.modules
WORK = Path("/kaggle/working") if ON_KAGGLE else Path("/content") if ON_COLAB else Path.cwd()
os.chdir(WORK)

CORES = os.cpu_count() or 1
smi = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True)
GPUS = smi.stdout.strip().splitlines() if smi.returncode == 0 else []
N_GPU = max(len(GPUS), 1) if LLM == "hf" else 1
ONLINE = reachable("https://huggingface.co")

print("platform :", "Kaggle" if ON_KAGGLE else "Colab" if ON_COLAB else "local")
print("python   :", sys.version.split()[0])
print("cpu cores:", CORES)
print("gpus     :", len(GPUS), GPUS)
print("internet :", ONLINE)

LLM_READY = LLM != "hf" or (len(GPUS) > 0 and ONLINE)
if not LLM_READY:
    print("\nThe model lanes will be skipped.")
    if not GPUS:
        print("  no GPU: set Accelerator to GPU T4 x2 in the side panel")
    if not ONLINE:
        print("  no internet: switch Internet on in the side panel")
stamp("environment checked")

platform : Kaggle
python   : 3.13.15
cpu cores: 4
gpus     : 2 ['Tesla T4, 15360 MiB', 'Tesla T4, 15360 MiB']
internet : True
[    0s] environment checked


## 2. Get the code

One file. The notebook uses a copy next to it, then an uploaded Kaggle dataset, then a clone of your repo.

In [3]:
EXPECTED_VERSION = "1.2.0"     # must match __version__ in the code_review_assistant.py you uploaded


def find_source():
    uploaded = sorted(Path("/kaggle/input").glob("**/code_review_assistant*.py"),
                      key=lambda p: p.stat().st_mtime, reverse=True)
    if uploaded:
        return uploaded[0]                  # newest upload wins if there are several
    here = WORK / "code_review_assistant.py"
    return here if here.exists() else None


source = find_source()
if source is None:
    if "<your-username>" in REPO_URL:
        raise RuntimeError("Set REPO_URL in the settings cell, or upload code_review_assistant.py "
                           "as a dataset (Add Input, then Upload).")
    if sh(f"git clone -q {REPO_URL} repo") != 0:
        raise RuntimeError(f"could not clone {REPO_URL}")
    source = WORK / "repo" / "code_review_assistant.py"

target = WORK / "code_review_assistant.py"
if source.resolve() != target.resolve():
    shutil.copy(source, target)             # always a clean name, so imports and the shell both work
print("using", source)

CRA = f"{sys.executable} code_review_assistant.py"
found_version = subprocess.run(f"{CRA} --version", shell=True, capture_output=True,
                               text=True).stdout.split()[-1]
print("code_review_assistant", found_version)
if found_version != EXPECTED_VERSION:
    # Kaggle pins each input to the dataset version that was current when it was
    # added, so uploading a new version does not reach the notebook by itself.
    raise RuntimeError(
        f"This is version {found_version} of code_review_assistant.py, but the notebook expects "
        f"{EXPECTED_VERSION}. Kaggle is still using an older upload. In the right-hand panel, open "
        "Input, find the dataset, and choose the newest version (the three-dot menu, or the "
        "'new version available' notice), then run again.")

using /kaggle/input/datasets/hansinees/dataset1/code_review_assistant (2).py
code_review_assistant 1.2.0


## 3. Tools

`ruff` covers what flake8, pycodestyle, pyflakes, bugbear, many of bandit's checks and a complexity score would each do separately, in one call that costs about ten milliseconds a file. `bandit` adds security checks of its own. `gradio` is for the UI check at the end.

In [4]:
sh(f"{sys.executable} -m pip install -q ruff bandit gradio")

missing = [m for m in ("transformers", "accelerate", "bitsandbytes") if importlib.util.find_spec(m) is None]
if missing:
    sh(f"{sys.executable} -m pip install -q {' '.join(missing)}")

os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"   # faster model downloads
print("ruff  :", shutil.which("ruff"))
print("bandit:", shutil.which("bandit"))
stamp("tools installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.7/134.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 48.6 MB/s eta 0:00:00
ruff  : /usr/local/bin/ruff
bandit: /usr/local/bin/bandit
[   14s] tools installed


## 4. Lane A: start the model downloads

Started first because it is the slowest step and needs nothing from the CPU.

In [5]:
downloads = {}

if LLM_READY and LLM == "hf":
    log = open("download.log", "w")

    def start_download(repo_id):
        code = f"from huggingface_hub import snapshot_download; print(snapshot_download({repo_id!r}))"
        downloads[repo_id] = subprocess.Popen([sys.executable, "-c", code],
                                              stdout=log, stderr=subprocess.STDOUT)

    start_download(MODEL)
    if RUN_7B:
        start_download(MODEL_7B)
    stamp("lane A: model downloads running in the background")
else:
    print("lane A skipped")

[   14s] lane A: model downloads running in the background


## 5. Lane B: CPU work, while the models download

The corpus has 40 cases. 30 hold defects the rules are built to catch (plus clean controls and edge cases). 10 hold logic errors that no rule can see: wrong boundaries, off-by-one loops, missing cases, a file never closed. A static-only run should miss all 10. That is deliberate. Whatever the model finds there is its real contribution.

The `#@ bug` comments that label each case are stripped before review, so neither the rules nor the model can see the answer key.

In [6]:
sh(f"{CRA} selftest | grep -E 'FAIL|checks passed'")
sh(f"{CRA} dump-corpus samples")
sh(f"{CRA} eval -o eval_static.json")
stamp("lane B: self-test and static evaluation done")

60/60 checks passed
wrote 40 files to samples/
Backend        : static only
Cases          : 40 (crashes: 0)
Match tolerance: +/-1 line(s)

category             P       R      F1    TP    FP    FN
--------------------------------------------------------
OVERALL          1.000   0.778   0.875    35     0    10
bug              1.000   0.565   0.722    13     0    10
security         1.000   1.000   1.000     9     0     0
style            1.000   1.000   1.000     8     0     0
optimization     1.000   1.000   1.000     5     0     0
--------------------------------------------------------
rule-visible     1.000   1.000   1.000    35     0     0
rule-blind       0.000   0.000   0.000     0     0    10

Latency: mean 6.8 ms, median 7.7 ms, p95 15.8 ms, max 16.6 ms
Throughput: 1580 lines/second per worker
Wall clock: 0.3s with 1 worker(s), including start-up and model loading

wrote eval_static.json
[   21s] lane B: self-test and static evaluation done


### Benchmark on real code, one core versus all of them

In [7]:
timings = {}
for workers in sorted({1, CORES}):
    sh(f"{CRA} benchmark --workers {workers} -o bench_{workers}w.json > /dev/null")
    timings[workers] = json.load(open(f"bench_{workers}w.json"))

print(f"{'workers':>8} {'wall (s)':>9} {'lines/s':>10} {'findings':>9}")
for w, data in timings.items():
    print(f"{w:>8} {data['wall_seconds']:>9.1f} {data['lines_per_second']:>10,.0f} {data['total_findings']:>9,}")
if CORES > 1:
    print(f"\nspeedup with {CORES} workers: "
          f"{timings[1]['wall_seconds'] / timings[CORES]['wall_seconds']:.2f}x, same findings either way")

shutil.copy(f"bench_{CORES}w.json", "benchmark_results.json")
data = timings[CORES]
print(f"\n{data['files_scanned']} files, {data['total_lines']:,} lines: "
      f"{data['findings_per_100_lines']} findings per 100 lines, "
      f"{data['serious_per_100_lines']} critical or high per 100 lines")
stamp("lane B: stdlib benchmark done")

 workers  wall (s)    lines/s  findings
       1     130.1      2,327     5,624
       4      16.1     18,750     5,624

speedup with 4 workers: 8.06x, same findings either way

599 files, 302,717 lines: 1.86 findings per 100 lines, 0.064 critical or high per 100 lines
[  168s] lane B: stdlib benchmark done


### Ruff and bandit adapters

In [8]:
sys.path.insert(0, str(WORK))
from code_review_assistant import ExternalTools, SourceUnit

probe = SourceUnit("import os\nimport sys\n\n\ndef f(x=[]):\n    eval(x)\n", "probe.py")
for tool in ("ruff", "bandit"):
    if not shutil.which(tool):
        print(f"{tool}: not installed, skipped")
        continue
    found = ExternalTools(enabled=True, tools=(tool,)).run(probe)
    print(f"{tool}: {len(found)} findings -> {[f.rule for f in found][:6]}")
    if not found:
        print(f"  WARNING: {tool} is installed but returned nothing; its output format may have changed")

if shutil.which("ruff") or shutil.which("bandit"):
    sh(f"{CRA} review samples --external-tools --format json -o review_tools.json")
    data = json.load(open("review_tools.json"))
    rules = [f["rule"] for item in data["files"] for f in item["findings"]]
    print(f"\ncorpus review with the tools on: {len(rules)} findings, "
          f"{sum(r.startswith('ruff:') for r in rules)} ruff-only, "
          f"{sum(r.startswith('bandit:') for r in rules)} bandit-only "
          "(anything our rules already cover is dropped, not shown twice)")

ruff: 4 findings -> ['ruff:F401', 'ruff:F401', 'ruff:B006', 'ruff:S307']
bandit: 1 findings -> ['bandit:B307']
wrote review_tools.json

corpus review with the tools on: 42 findings, 3 ruff-only, 3 bandit-only (anything our rules already cover is dropped, not shown twice)


### Precision audit on code nobody here wrote

The corpus was written by the same person as the rules, so a perfect score there proves little. This samples findings from the Python standard library instead and writes `audit_sheet.csv`. Open it, mark the `verdict` column **real**, **false** or **unsure** for each row, and the share marked real is the tool's precision on outside code.

An earlier audit of 50 findings (seed 7) found 26% precision, with three rules causing most of the false alarms: unused variables in tuple unpacking, `in` followed by indexing, and `assert` on internal state. Those rules were narrowed. This audit uses a different sample, so the new figure is not measured on the rows the fix was tuned to.

In [9]:
import random
import sysconfig

stdlib = Path(sysconfig.get_paths()["stdlib"])
pool = [p for p in sorted(stdlib.rglob("*.py"))
        if not {"test", "tests", "idlelib", "site-packages"} & set(p.parts)]
random.seed(11)   # a fresh sample: seed 7's rows were used to tune three rules
picked = random.sample(pool, min(80, len(pool)))

sh(f"{CRA} review {' '.join(str(p) for p in picked)} --format json -o audit_review.json > /dev/null")
data = json.load(open("audit_review.json"))

by_category = {}
for item in data["files"]:
    for f in item["findings"]:
        by_category.setdefault(f["category"], []).append((item["path"], f))
for rows in by_category.values():
    random.shuffle(rows)

sample = []
while len(sample) < 50 and any(by_category.values()):
    for rows in by_category.values():            # take one from each category in turn
        if rows and len(sample) < 50:
            sample.append(rows.pop())


def context(path, line, radius=2):
    lines = Path(path).read_text(errors="replace").splitlines()
    low, high = max(line - 1 - radius, 0), min(line + radius, len(lines))
    return "\n".join(f"{n + 1:>4}  {lines[n]}" for n in range(low, high))


with open("audit_sheet.csv", "w", newline="") as fh:
    writer = csv.writer(fh)
    writer.writerow(["file", "line", "category", "rule", "severity", "issue", "code", "verdict", "comment"])
    for path, f in sorted(sample, key=lambda pair: (pair[1]["category"], pair[0], pair[1]["line"])):
        writer.writerow([Path(path).name, f["line"], f["category"], f["rule"], f["severity"],
                         f["message"], context(path, f["line"]), "", ""])

counts = {c: sum(f["category"] == c for _, f in sample) for c in by_category}
print(f"{len(picked)} files reviewed, {sum(len(i['findings']) for i in data['files'])} findings in total")
print(f"audit_sheet.csv: {len(sample)} findings sampled, by category: {counts}")
stamp("lane B: audit sheet written")

80 files reviewed, 851 findings in total
audit_sheet.csv: 50 findings sampled, by category: {'style': 13, 'optimization': 13, 'security': 12, 'bug': 12}
[  185s] lane B: audit sheet written


## 6. Wait for the models

In [10]:
def cache_gb(repo_id):
    try:
        from huggingface_hub.constants import HF_HUB_CACHE
    except ImportError:
        HF_HUB_CACHE = str(Path.home() / ".cache/huggingface/hub")
    folder = Path(HF_HUB_CACHE) / ("models--" + repo_id.replace("/", "--"))
    return sum(p.stat().st_size for p in folder.rglob("*") if p.is_file()) / 1e9


deadline = time.time() + 3600
while any(p.poll() is None for p in downloads.values()):
    if time.time() > deadline:
        raise TimeoutError("model download is taking over an hour; check download.log")
    sizes = ", ".join(f"{m.split('/')[-1]} {cache_gb(m):.1f} GB" for m, p in downloads.items() if p.poll() is None)
    print(f"[{time.time() - T0:5.0f}s] downloading: {sizes}")
    time.sleep(15)

failed = [m for m, p in downloads.items() if p.returncode != 0]
if failed:
    print(open("download.log").read()[-2000:])
    raise RuntimeError(f"download failed for {failed}")
if downloads:
    os.environ["HF_HUB_OFFLINE"] = "1"      # everything is cached; no more network calls
stamp("lane A: model files are on disk")

[  185s] lane A: model files are on disk


## 7. Lane C: the models

### Smoke test

Two files, one GPU. If the model does not return JSON the pipeline can read, this is where it shows.

In [11]:
if LLM_READY:
    sh(f"{CRA} review samples/sql_injection.py samples/off_by_one_sum.py --llm {LLM} --model {MODEL} "
       "--workers 1 --format json -o smoke.json", env={"CUDA_VISIBLE_DEVICES": "0"})
    data = json.load(open("smoke.json"))
    findings = [f for item in data["files"] for f in item["findings"]]
    explained = sum(f["source"] == "static+llm" for f in findings)
    own = [f for f in findings if f["source"] == "llm"]
    print(f"\nexplained {explained} static findings, proposed {len(own)} of its own")
    for f in own:
        print(f"  line {f['line']}: {f['message']} (confidence {f['confidence']})")
    for item in data["files"]:
        for e in item["errors"]:
            print("  note:", e)
else:
    print("skipped")

wrote smoke.json

explained 2 static findings, proposed 0 of its own


### Score the 3B model

The corpus is split across the GPUs, one copy of the model per GPU. Per-case timings leave out model loading, so the latency is the model working, not starting.

In [12]:
if LLM_READY:
    sh(f"{CRA} eval --llm {LLM} --model {MODEL} --workers {N_GPU} --gpus auto --show-errors -o eval_llm.json")
    stamp("lane C: 3B evaluation done")
else:
    print("skipped")

using 2 worker processes on GPU(s) [0, 1]
Backend        : hf:Qwen/Qwen2.5-Coder-3B-Instruct
Cases          : 40 (crashes: 0)
Match tolerance: +/-1 line(s)

category             P       R      F1    TP    FP    FN
--------------------------------------------------------
OVERALL          1.000   0.778   0.875    35     0    10
bug              1.000   0.565   0.722    13     0    10
security         1.000   1.000   1.000     9     0     0
style            1.000   1.000   1.000     8     0     0
optimization     1.000   1.000   1.000     5     0     0
--------------------------------------------------------
rule-visible     1.000   1.000   1.000    35     0     0
rule-blind       0.000   0.000   0.000     0     0    10

Latency: mean 5081.4 ms, median 5201.2 ms, p95 11840.7 ms, max 14559.9 ms
Throughput: 2 lines/second per worker
Wall clock: 123.5s with 2 worker(s), including start-up and model loading

LLM stage: explained 32 of 35 static findings; added 0 of its own, 0 of them wrong
Ex

### Score the 7B model, 4-bit

Quantised to about 5 GB so one copy fits on each T4. This is the model-selection comparison.

In [13]:
if LLM_READY and RUN_7B and LLM == "hf":
    sh(f"{CRA} eval --llm hf --model {MODEL_7B} --quantize 4bit --workers {N_GPU} --gpus auto "
       "--show-errors -o eval_llm_7b.json")
    stamp("lane C: 7B evaluation done")
else:
    print("skipped")

using 2 worker processes on GPU(s) [0, 1]
Backend        : hf:Qwen/Qwen2.5-Coder-7B-Instruct (4bit)
Cases          : 40 (crashes: 0)
Match tolerance: +/-1 line(s)

category             P       R      F1    TP    FP    FN
--------------------------------------------------------
OVERALL          0.974   0.844   0.905    38     1     7
bug              0.941   0.696   0.800    16     1     7
security         1.000   1.000   1.000     9     0     0
style            1.000   1.000   1.000     8     0     0
optimization     1.000   1.000   1.000     5     0     0
--------------------------------------------------------
rule-visible     1.000   1.000   1.000    35     0     0
rule-blind       0.750   0.300   0.429     3     1     7

Latency: mean 7562.9 ms, median 7721.8 ms, p95 13488.6 ms, max 18075.7 ms
Throughput: 1 lines/second per worker
Wall clock: 223.2s with 2 worker(s), including start-up and model loading

LLM stage: explained 35 of 35 static findings; added 4 of its own, 1 of them w

### Static only versus each model

This is the table for `REPORT.md`. Read three rows in particular. **Rule-blind recall** is what the model finds that the rules cannot. **Precision of the model's own findings** is whether it invents problems. On the rule-blind cases a finding counts only if it is on the labelled line *and* its explanation names the actual bug (each case lists a few keywords). Pointing at the right line for the wrong reason counts as wrong. **Specific explanations** is whether its advice names the actual code.

In [14]:
from code_review_assistant import format_eval_markdown

runs = {"static only": "eval_static.json", "3B": "eval_llm.json", "7B 4-bit": "eval_llm_7b.json"}
results = {name: json.load(open(path)) for name, path in runs.items() if Path(path).exists()}


def own_precision(r):
    added = r["llm"]["added"]
    return f"{(added - r['llm']['added_wrong']) / added:.3f}" if added else "-"


def specific_share(r):
    n = r["llm"]["enriched"]
    return f"{r['llm']['specific'] / n:.0%}" if n else "-"


rows = [
    ("overall F1", lambda r: f"{r['overall']['f1']:.3f}"),
    ("rule-visible F1", lambda r: f"{r['subsets']['static_visible']['f1']:.3f}"),
    ("rule-blind recall", lambda r: f"{r['subsets']['static_blind']['recall']:.3f}"),
    ("own findings kept", lambda r: str(r["llm"]["added"])),
    ("own findings precision", own_precision),
    ("static findings explained", lambda r: f"{r['llm']['enriched']}/{r['llm']['static_findings']}"),
    ("specific explanations", specific_share),
    ("mean latency (ms)", lambda r: f"{r['latency']['mean_seconds'] * 1000:,.1f}"),
    ("wall clock (s)", lambda r: f"{r['latency']['wall_seconds']:.0f}"),
]

names = list(results)
print(f"{'':<27}" + "".join(f"{n:>14}" for n in names))
print("-" * (27 + 14 * len(names)))
for label, get in rows:
    print(f"{label:<27}" + "".join(f"{get(results[n]):>14}" for n in names))

table = ["| | " + " | ".join(names) + " |", "| --- |" + " ---: |" * len(names)]
for label, get in rows:
    table.append(f"| {label} | " + " | ".join(get(results[n]) for n in names) + " |")
with open("model_comparison.md", "w") as fh:
    fh.write("\n".join(table) + "\n")
    for name in names[1:]:
        fh.write(f"\n### {name}\n\n{format_eval_markdown(results[name])}\n")
print("\nwrote model_comparison.md (tables ready for REPORT.md)")

                              static only            3B      7B 4-bit
---------------------------------------------------------------------
overall F1                          0.875         0.875         0.905
rule-visible F1                     1.000         1.000         1.000
rule-blind recall                   0.000         0.000         0.300
own findings kept                       0             0             4
own findings precision                  -             -         0.750
static findings explained            0/35         32/35         35/35
specific explanations                   -           78%           71%
mean latency (ms)                     6.8       5,081.4       7,562.9
wall clock (s)                          0           123           223

wrote model_comparison.md (tables ready for REPORT.md)


### Rating sheet for the expert-review metric

The brief asks for optimisation suggestions to be *rated by expert reviewers*. A program cannot do that honestly, so this writes `rating_sheet.csv`: up to 20 of the model's explanations and fixes, optimisation ones first, with an empty column for a 1 to 5 score. They come from the 7B model, the one the comparison above recommends. Fill it in yourself or with your mentor, then put the average in the report.

In [15]:
# The examples people rate should come from the model the comparison recommends.
if RUN_7B and LLM == "hf":
    REVIEW_ARGS, REVIEW_NAME = f"--llm hf --model {MODEL_7B} --quantize 4bit", "7B 4-bit"
else:
    REVIEW_ARGS, REVIEW_NAME = f"--llm {LLM} --model {MODEL}", MODEL.split("/")[-1]

if LLM_READY:
    sh(f"{CRA} review samples {REVIEW_ARGS} --workers {N_GPU} --gpus auto "
       "--format json -o review_llm_samples.json")
    data = json.load(open("review_llm_samples.json"))
    rated = [(item["path"], f) for item in data["files"] for f in item["findings"]
             if f["source"] in ("static+llm", "llm")]
    rated.sort(key=lambda pair: (pair[1]["category"] != "optimization", pair[0], pair[1]["line"]))
    rated = rated[:20]

    with open("rating_sheet.csv", "w", newline="") as fh:
        writer = csv.writer(fh)
        writer.writerow(["file", "line", "category", "rule", "issue", "explanation", "fix",
                         "usefulness_1_to_5", "comment"])
        for path, f in rated:
            writer.writerow([Path(path).name, f["line"], f["category"], f["rule"], f["message"],
                             f["rationale"], f["suggestion"], "", ""])
    print(f"wrote rating_sheet.csv with {len(rated)} rows from the {REVIEW_NAME} model "
          f"({sum(f['category'] == 'optimization' for _, f in rated)} optimisation suggestions)")
    stamp("lane D: rating sheet written")
else:
    print("skipped")

using 2 worker processes on GPU(s) [0, 1]
wrote review_llm_samples.json
wrote rating_sheet.csv with 20 rows from the 7B 4-bit model (5 optimisation suggestions)
[  774s] lane D: rating sheet written


### Batch mode on real files, in parallel

In [16]:
if LLM_READY and RUN_BATCH_DEMO:
    import sysconfig

    stdlib = Path(sysconfig.get_paths()["stdlib"])
    files = " ".join(str(p) for p in sorted((stdlib / "json").glob("*.py"))[:6])

    start = time.time()
    sh(f"{CRA} review {files} {REVIEW_ARGS} --workers {N_GPU} --gpus auto "
       "--min-severity medium --format markdown -o batch_review.md")
    text = open("batch_review.md").read()
    print(f"\n{time.time() - start:.0f}s wall clock with {N_GPU} worker(s), {REVIEW_NAME} model")
    print("model output that could not be parsed:", text.count("Could not parse model output"), "chunk(s)")
    print("\n" + "".join(text.splitlines(True)[:30]))
    stamp("lane D: batch demo done")
else:
    print("skipped")

using 2 worker processes on GPU(s) [0, 1]
wrote batch_review.md

173s wall clock with 2 worker(s), 7B 4-bit model
model output that could not be parsed: 0 chunk(s)

# Code review

25 findings across 5 file(s).

## `/usr/lib/python3.13/json/__init__.py`

| Line | Severity | Category | Rule | Issue |
| ---: | --- | --- | --- | --- |
| 120 | medium | style | `Y005` | 'dump' takes 11 parameters (limit 5) |
| 120 | medium | style | `Y007` | 'dump' has cyclomatic complexity 13 (limit 10) |
| 185 | medium | style | `Y005` | 'dumps' takes 10 parameters (limit 5) |
| 185 | medium | style | `Y007` | 'dumps' has cyclomatic complexity 12 (limit 10) |
| 248 | medium | style | `Y007` | 'detect_encoding' has cyclomatic complexity 13 (limit 10) |
| 278 | medium | style | `Y005` | 'load' takes 7 parameters (limit 5) |
| 304 | medium | style | `Y005` | 'loads' takes 7 parameters (limit 5) |
| 304 | medium | style | `Y007` | 'loads' has cyclomatic complexity 17 (limit 10) |

### Details

**Line 120 — 'du

## 8. Interactive UI

The UI is built and its review function is called directly, which checks the whole path the button uses without blocking the run. Then it is started in the background for a few seconds to confirm it serves. Set `RUN_UI_LIVE = True` in an interactive session to keep it open with a public link.

In [17]:
if importlib.util.find_spec("gradio") is None:
    print("gradio not installed, UI check skipped")
else:
    from code_review_assistant import EXAMPLE_CODE, build_ui

    try:
        demo, review_fn = build_ui()                 # static backend: fast and needs no GPU
        summary, metrics, report_md = review_fn(EXAMPLE_CODE, ["bug", "security", "style", "optimization"], "info")
        print("UI review function:", summary.splitlines()[0])

        demo.launch(prevent_thread_lock=True, share=False, quiet=True)
        time.sleep(3)
        print("UI served at:", demo.local_url)
        demo.close()
        stamp("UI checked")
    except Exception as exc:
        # reported, not raised, so a gradio version quirk cannot stop the run before the summary
        print(f"UI check FAILED: {type(exc).__name__}: {exc}")

    if RUN_UI_LIVE:
        sh(f"{CRA} ui --share --llm {LLM} --model {MODEL}")

UI review function: **3 findings** in 3 ms (static 3 ms, LLM 0 ms) — backend: static only


UI served at: http://127.0.0.1:7860/
Closing server running on port: 7860
[  955s] UI checked


## 9. What happened, and where the files are

In [18]:
stamp("finished")
print("\ntimeline")
for seconds, label in TIMELINE:
    print(f"  {seconds:6.0f}s  {label}")

print("\nfiles written to", WORK)
for pattern in ("*.json", "*.md", "*.csv", "*.log"):
    for p in sorted(WORK.glob(pattern)):
        print(f"  {p.name:<28}{p.stat().st_size:>10,} bytes")
if ON_KAGGLE:
    print("\nAfter Save & Run All, download these from the version's Output tab.")

[  955s] finished

timeline
       0s  environment checked
      14s  tools installed
      14s  lane A: model downloads running in the background
      21s  lane B: self-test and static evaluation done
     168s  lane B: stdlib benchmark done
     185s  lane B: audit sheet written
     185s  lane A: model files are on disk
     375s  lane C: 3B evaluation done
     598s  lane C: 7B evaluation done
     774s  lane D: rating sheet written
     947s  lane D: batch demo done
     955s  UI checked
     955s  finished

files written to /kaggle/working
  audit_review.json              476,533 bytes
  bench_1w.json                      851 bytes
  bench_4w.json                      850 bytes
  benchmark_results.json             850 bytes
  eval_llm.json                   37,726 bytes
  eval_llm_7b.json                38,598 bytes
  eval_static.json                37,462 bytes
  review_llm_samples.json         50,898 bytes
  review_tools.json               47,625 bytes
  smoke.json            